# 第1章 交通问题与数据方案

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch01-design-v1`  
**必做：** 研究问题操作化；变量与数据适配审查  
**对象：** 明确到对象、位置与时间尺度  
**样本：** 沿用4类真实来源，不拼接为一个虚构总体  
**划分：** 方案设计，不进行预测训练  
**比较：** 同一交通问题至少比较两种来源，不按记录数评优

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 读取真实来源及观测定义
请解释：记录少是否必然说明需求小？任选两种来源，比较观测对象和限制。


In [ ]:
sources = pd.read_csv(ROOT/'chapters/data/ch01_sources.csv')
display(sources)


## 2. 直接查看数据样本
这不是把数据拼成一个总体。请比较事故一行与出租车聚合一行分别代表什么。


In [ ]:
snapshots = {}
for name in ['audit', 'crashes', 'taxi', 'sind']:
    item = json.loads((ROOT/f'projects/data/{name}.json').read_text(encoding='utf-8'))
    snapshots[name] = {'rows': len(item['rows']), 'first_observation': item['rows'][0]}
display(pd.DataFrame(snapshots).T)


## 3. 写出问题-指标-字段关系
修改下方列表，将示例问题改成自己的具体任务。没有相应字段的指标不得列为已测得。


In [ ]:
plan = pd.DataFrame([
    ['事故排查', '2024年1月哪些位置需优先核查', '事故记录数', 'crashes.rows: id/lat/lon', '不能直接得出风险率'],
    ['出租车运营', '同一区域夜间日均服务量如何变化', '上车记录条/小时/日', 'taxi.rows: date/borough/hour/count', '未观测未满足需求']
], columns=['scenario', 'question', 'indicator', 'observed_fields', 'boundary'])
display(plan)
csv_file(ROOT/'outputs/ch01/problem_variables.csv', plan.columns, plan.values)


## 4. 导出方案证据并完成文字论证
需提交两种来源适配对照、许可清单和最低可行方案。下面只生成证据与待回答问题，不伪造已开展的调查。


In [ ]:
report(ROOT, 1, '交通数据分析最低可行方案', {'来源数量': len(sources), '问题变量表': plan.to_string(index=False)}, ['为什么选择该对象与尺度？', '哪些判断仍需补充数据？', '如何合规获取和验证？'])
